# Style Prefix Demo (GPT-2 Nano)
This notebook installs Levanter from the `feat/style-prefix-token` branch,
creates a tiny chat dataset with `style` labels, inspects the resulting
prefix tokens and loss mask, and runs a short GPT-2 "nano" training loop.

In [ ]:
%%bash
pip install --quiet \
  git+https://github.com/chris544460/levanter.git@feat/style-prefix-token \
  datasets wandb draccus jax jaxlib

git clone https://github.com/chris544460/levanter.git
cd levanter
git checkout feat/style-prefix-token

mkdir -p data/style_demo
cat <<'EOF' > data/style_demo/train.jsonl
{"messages": [{"role": "system", "content": "style=wiki"}, {"role": "user", "content": "Who wrote the Odyssey?"}, {"role": "assistant", "content": "Homer wrote the Odyssey."}], "style": "wiki"}
{"messages": [{"role": "system", "content": "style=books"}, {"role": "user", "content": "Recommend a fantasy series."}, {"role": "assistant", "content": "Try The Wheel of Time by Robert Jordan."}], "style": "books"}
{"messages": [{"role": "system", "content": "style=news"}, {"role": "user", "content": "Summarize today's headlines."}, {"role": "assistant", "content": "Major markets rallied; new policies were announced."}], "style": "news"}
{"messages": [{"role": "system", "content": "style=wiki"}, {"role": "user", "content": "What is photosynthesis?"}, {"role": "assistant", "content": "Photosynthesis converts light energy into chemical energy."}], "style": "wiki"}
EOF

In [ ]:
%%bash
cd /content/levanter
pip uninstall -y levanter

pip install -e .

In [ ]:
chat_template = (
    "{%- for message in messages -%}\n"
    "{{ message['role'] }}: {{ message['content'] }}\n"
    "{%- endfor -%}\n"
    "{%- if add_generation_prompt %}assistant:{% endif %}"
)

format_cfg = ChatLmDatasetFormat(
    messages_field="messages",
    single_turn=False,
    chat_template=chat_template,
    pack=True,
    mask_user_turns=False,
    style_prefix=StylePrefixConfig(
        prefix_token="<style>",
        suffix_token="</style>",
        style_field="style",
    ),
)

processor = preprocessor_for_format(format_cfg, tokenizer)
processed = processor(entries[:2])

for idx, example in enumerate(processed):
    print(f"Example {idx}")
    print("input_ids:", example["input_ids"][:20])
    print("assistant_masks:", example["assistant_masks"][:20])
    tokens = tokenizer.convert_ids_to_tokens(example["input_ids"][:20])
    print("tokens:", tokens)
    print()


In [ ]:
%%bash
cd /content/levanter
PYTHONPATH=$PWD python -m levanter.main.train_lm --config_path config/gpt2_nano_style.yaml